In [5]:
## tools

from langchain_community.tools import ArxivQueryRun,WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper,ArxivAPIWrapper

In [6]:
api_wrapper_arxiv=ArxivAPIWrapper(top_k_results=2,doc_content_chars_max=500)
arxiv=ArxivQueryRun(api_wrapper=api_wrapper_arxiv,description="Query arxiv papers")
print(arxiv.name)

arxiv


In [7]:
arxiv.invoke("Attention is all you need")

'Published: 2021-05-06\nTitle: Do You Even Need Attention? A Stack of Feed-Forward Layers Does Surprisingly Well on ImageNet\nAuthors: Luke Melas-Kyriazi\nSummary: The strong performance of vision transformers on image classification and other vision tasks is often attributed to the design of their multi-head attention layers. However, the extent to which attention is responsible for this strong performance remains unclear. In this short report, we ask: is the attention layer even necessary? Specifi'

In [8]:
api_wrapper_wiki=WikipediaAPIWrapper(top_k_results=1, doc_content_chars_max=500)
wiki=WikipediaQueryRun(api_wrapper=api_wrapper_wiki)
wiki.name

'wikipedia'

In [9]:
wiki.invoke("Attention is all you need")

'Page: Attention Is All You Need\nSummary: "Attention Is All You Need" is a 2017 research paper on machine learning authored by eight scientists and engineers working at Google. The paper introduced a new deep learning architecture known as the transformer, based on the attention mechanism proposed in 2014 by Bahdanau et al. The transformer approach it describes has become the main architecture of a wide variety of artificial intelligence systems, including large language models. At the time, the '

In [10]:
# integrate tools in workflow
from dotenv import load_dotenv
load_dotenv()

import os

os.environ["TAVILY_API_KEY"]=os.getenv("TAVILY_API_KEY")
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")

In [11]:
# tavily search tool
from langchain_community.tools.tavily_search import TavilySearchResults

tavily = TavilySearchResults()

In [12]:
tavily.invoke("Provide me the recent AI new ?s")

[{'title': 'The Latest AI News and Breakthroughs That Matter Most',
  'url': 'https://www.crescendo.ai/news/latest-ai-news-and-updates',
  'content': "Date: July 2, 2026\n\nSummary: Anthropic is in early discussions with Samsung to develop a custom AI chip, joining competitors seeking to reduce reliance on Nvidia. The move follows OpenAI's announcement of its own custom inference processor built with Broadcom. Multiple AI companies now pursue specialized silicon to optimize performance and address compute constraints.\n\nSource: TechCrunch\n\n### (AI Update) New York Enacts First-of-Its-Kind Law Requiring AI-Generated Ad Disclosures\n\nDate: July 2, 2026 [...] Source: Washington Post\n\n### (AI Controversy) Google Files AI Phishing Lawsuit in New York\n\nDate: June 12, 2026\n\nSummary: Google filed a major New York lawsuit against operators of an AI-powered phishing scheme that used artificial intelligence to impersonate businesses and deceive users into sharing sensitive information. 

In [13]:
## combine all these tools in the list
tools=[arxiv, wiki, tavily]

In [16]:
## Initialize the LLM model
from langchain_groq import ChatGroq

llm = ChatGroq(model = "qwen/qwen3.8-27b")

In [17]:
llm.invoke("What is AI ?")

AIMessage(content='**AI**, or **Artificial Intelligence**, is a branch of computer science focused on creating systems capable of performing tasks that typically require human intelligence.\n\nRather than following a fixed set of rules, AI systems are designed to learn from data, adapt to new inputs, and make decisions or predictions with minimal human intervention.\n\n### Key Components of AI:\n1.  **Machine Learning (ML):** A subset of AI where algorithms automatically improve through experience. Instead of being explicitly programmed, the system analyzes data to find patterns.\n2.  **Deep Learning:** A further subset of ML using neural networks (inspired by the human brain) to handle complex tasks like image recognition, natural language processing, and speech synthesis.\n3.  **Natural Language Processing (NLP):** Enables machines to understand, interpret, and generate human language (e.g., chatbots, translation tools).\n4.  **Computer Vision:** Allows machines to "see" and interpre

In [19]:
## Combine with our llm
llm_with_tools= llm.bind_tools(tools=tools)

In [20]:
## Execute this call
llm_with_tools.invoke("What is the recent news on AI ?")

AIMessage(content="I'll search for recent news on AI for you.\n\n", additional_kwargs={'tool_calls': [{'id': '8fbr3gwsb', 'function': {'arguments': '{"query":"recent AI news"}', 'name': 'tavily_search_results_json'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 43, 'prompt_tokens': 478, 'total_tokens': 521, 'completion_time': 0.108744197, 'completion_tokens_details': None, 'prompt_time': 0.041063879, 'prompt_tokens_details': None, 'queue_time': 0.023832543, 'total_time': 0.149808076}, 'model_name': 'qwen/qwen3.8-27b', 'system_fingerprint': 'fp_43dd1ea4ee', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None}, id='run--01a0d53e-aa90-7690-bf24-c3163d029371-0', tool_calls=[{'name': 'tavily_search_results_json', 'args': {'query': 'recent AI news'}, 'id': '8fbr3gwsb', 'type': 'tool_call'}], usage_metadata={'input_tokens': 478, 'output_tokens': 43, 'total_tokens': 521})

In [22]:
llm_with_tools.invoke("What is the latest research on quantum computing ?")

AIMessage(content='### Latest Research on Quantum Computing\n\nTo provide an accurate and up-to-date overview of the latest research in quantum computing, I will search for recent developments and state-of-the-art papers.\n\n', additional_kwargs={'tool_calls': [{'id': 'nh5n6hq6s', 'function': {'arguments': '{"query":"latest research on quantum computing 2024 2025 breakthroughs"}', 'name': 'tavily_search_results_json'}, 'type': 'function'}, {'id': 'gpp1tsnca', 'function': {'arguments': '{"query":"quantum computing recent advances"}', 'name': 'arxiv'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 113, 'prompt_tokens': 479, 'total_tokens': 592, 'completion_time': 0.326589812, 'completion_tokens_details': None, 'prompt_time': 0.041143272, 'prompt_tokens_details': None, 'queue_time': 0.018443594, 'total_time': 0.367733084}, 'model_name': 'qwen/qwen3.8-27b', 'system_fingerprint': 'fp_43dd1ea4ee', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'l

In [23]:
llm_with_tools.invoke("What is machine learning ?")

AIMessage(content='Machine learning is a branch of artificial intelligence (AI) that enables computers to learn from data and improve their performance on tasks without being explicitly programmed for every specific rule. Instead of following hand-coded instructions, machine learning algorithms identify patterns in data and use those patterns to make predictions, decisions, or classifications.\n\nHere are the key aspects of machine learning:\n\n1. **Data-Driven**: ML systems rely on large datasets to train models. The more relevant data available, the better the model typically performs.\n\n2. **Types of Learning**:\n   - **Supervised Learning**: The model is trained on labeled data (input-output pairs), such as classifying emails as spam or not spam.\n   - **Unsupervised Learning**: The model finds hidden patterns or groupings in unlabeled data, such as customer segmentation.\n   - **Reinforcement Learning**: The model learns by interacting with an environment and receiving rewards or